# 01 - Entrenamiento, evaluación e inferencia LSC5

Reproducción del candidato fijado `v4_contact_bimanual`: inventario de train, validación LOSO por persona, reentrenamiento de tres MLP y comparación con los artefactos originales.

**Requisitos:** Python 3.10, numpy 1.26.4 y torch 2.4.1 CPU; instalación del paquete según el README. Para abrir este archivo instala Jupyter en tu entorno de trabajo y selecciona ese Python como kernel. El notebook no contiene datos ni pesos: necesitas acceso autorizado a `train/landmarks/sign_XX/*.npy` y al bundle `bundle_iter4_v1` (incluye `train_reference.npz`). Define `LSC5_DATA_ROOT` y `LSC5_BUNDLE` si no usas las rutas predeterminadas.

La inferencia test está **desactivada por defecto**. Habilítala solo con autorización para usar el modelo ya fijado; requiere además `test/landmarks/`, `test/submission_template.csv` y el CSV de referencia (`LSC5_REFERENCE_CSV`). No usar test para ajuste, calibración ni selección.

Los archivos generados quedan en `artifacts/notebook_reproduction/`, fuera de Git. El notebook público no incluye salidas por muestra. Resultados registrados y limitaciones: `docs/07_results_and_publication_protocol.md`; no hay puntuación oficial confirmada.


In [ ]:
import os, sys, json, time, shutil, hashlib
from pathlib import Path
import numpy as np

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
assert ROOT is not None, "Ejecutar desde el repositorio o su carpeta notebooks/"
DATA_ROOT = Path(os.environ.get("LSC5_DATA_ROOT", str(ROOT / "data"))).expanduser().resolve()
HERE = ROOT
WORK = ROOT / "artifacts" / "notebook_reproduction"
DELIVERED_BUNDLE = Path(os.environ.get("LSC5_BUNDLE", str(ROOT / "artifacts" / "iter3" / "bundle_iter4_v1"))).expanduser().resolve()
DELIVERED_CSV = Path(os.environ.get("LSC5_REFERENCE_CSV", str(ROOT / "artifacts" / "submissions" / "iter4_v1" / "submission.csv"))).expanduser().resolve()
RUN_LOSO = True
RUN_TEST_INFERENCE = False  # Habilitar únicamente para inferencia autorizada del modelo fijado.

sys.path.insert(0, str(ROOT / "src"))
import torch
from lsc5 import wave, wave_iter
from lsc5.data import CLASSES, NAME, read_sequence
from lsc5.delivery import write_submission, validate_submission

assert (DATA_ROOT / "train" / "landmarks").is_dir(), "Revisar LSC5_DATA_ROOT"
assert (DELIVERED_BUNDLE / "train_reference.npz").is_file(), "Revisar acceso al bundle fijado y LSC5_BUNDLE"
WORK.mkdir(parents=True, exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
print("numpy", np.__version__, "| torch", torch.__version__, "| clases", CLASSES)


## 1. Inventario de entrenamiento

Cada nombre de archivo codifica seña, persona, vista y cámara (`S01_M02_V01_C01_Clip...`). La seña y la persona se usan **solo** como etiqueta y para separar los pliegues LOSO; nunca como rasgo. El orden se fija por identificador, igual que en el entrenamiento original, y se comprueba contra el orden guardado con los pesos.

In [ ]:
samples = []
for path in sorted((DATA_ROOT / "train" / "landmarks").rglob("*.npy"), key=lambda p: p.stem):
    m = NAME.fullmatch(path.stem)
    assert m and path.parent.name == CLASSES[int(m[1]) - 1], path.name
    samples.append({"id": path.stem, "label": path.parent.name, "signer": m[2], "view": m[3],
                    "landmarks_path": path.relative_to(DATA_ROOT / "train").as_posix()})
manifest = {"samples": samples,
            "manifest_hash": hashlib.sha256(json.dumps([s["id"] for s in samples]).encode()).hexdigest()}
ref_ids = np.load(DELIVERED_BUNDLE / "train_reference.npz")["ids"].tolist()
assert [s["id"] for s in samples] == ref_ids, "El inventario no coincide con el del entrenamiento original"
ids, xs, y, signers = wave.load_train(DATA_ROOT / "train", manifest)
print(len(ids), "clips;", {s: int((signers == s).sum()) for s in ("M02", "M03", "M04")},
      "| fotogramas por clip:", min(map(len, xs)), "a", max(map(len, xs)))

## 2. Representación

Para cada clip, de forma determinista y sin etiquetas:

1. **Lateralidad canónica.** Si la muñeca izquierda sube más que la derecha, se refleja el clip (`x → 1 − x`) y se intercambian izquierda y derecha.
2. **Compuerta de reposo.** Se anula una mano en los fotogramas en que está más de 1,3 anchos de hombro bajo los hombros y casi quieta.
3. **Tramo activo.** Se recorta el clip a los fotogramas con alguna muñeca levantada, más un margen de 3.
4. **Normalización corporal.** Coordenadas centradas en el punto medio de los hombros y escaladas por su ancho: 49 puntos globales, forma local de cada mano y 9 máscaras de presencia (191 valores por fotograma), más 18 distancias de forma de mano y 98 velocidades.
5. **Resumen temporal.** Media, desviación estándar y media de cada tercio (1.535 valores), más 4 rasgos de contacto con el pecho y 6 de actividad bimanual: **1.545 valores por clip**.

En entrenamiento se añaden 4 copias por clip con un cambio de cámara simulado (escorzo 0,7 a 1, rotación ±8°, zoom ±15%). El clasificador es un MLP (1.545 → 128 → 5, ReLU, dropout 0,3), AdamW, 80 épocas, lotes equilibrados por persona y clase, y la predicción promedia tres semillas (17, 29, 43).

In [ ]:
CFG = wave_iter.CANDIDATES["v4_contact_bimanual"]
face_perm = np.load(DELIVERED_BUNDLE / "face_perm.npy")   # mapeo de simetría facial; la cara no entra en los rasgos
canon = [wave.canonical(x, face_perm) for x in xs]
example = wave_iter.features(canon[0], CFG)
print("configuración:", CFG)
print("dimensión de rasgos:", example.shape[0])

## 3. Validación LOSO (opcional)

Tres pliegues: se entrena con dos personas (750 clips más sus copias aumentadas) y se evalúa con la tercera (375 clips, sin aumento). Normalización y pesos se ajustan solo con las personas de entrenamiento de cada pliegue. Resultado esperado: **1.108/1.125** (M02 373, M03 368, M04 367).

In [ ]:
if RUN_LOSO:
    t0 = time.time()
    n_aug = CFG["n_aug"]
    rng = np.random.default_rng(wave_iter.AUG_SEED)
    feats = [wave_iter.features(c, CFG) for c in canon]
    extra = [[wave_iter.features(c, CFG, rng, augment=True) for _ in range(n_aug)] for c in canon]
    per_seed = {}
    for seed in wave.SEEDS:
        probs = np.zeros((len(y), 5), np.float32)
        for held in ("M02", "M03", "M04"):
            fit = np.where(signers != held)[0]
            val = np.where(signers == held)[0]
            fx = [feats[i] for i in fit] + [extra[i][k] for k in range(n_aug) for i in fit]
            model = wave_iter.fit_mlp_cfg(fx, np.tile(y[fit], n_aug + 1), np.tile(signers[fit], n_aug + 1), seed)
            probs[val] = model([feats[i] for i in val])
        per_seed[seed] = probs
    summary, avg = wave.summarise(per_seed, y, signers)
    pred = avg.argmax(1)
    cm = np.zeros((5, 5), int)
    for a, b in zip(y, pred):
        cm[a, b] += 1
    print(f"LOSO (promedio de 3 semillas): {summary['avg_correct']}/1125 = {100 * summary['avg_accuracy']:.2f}%")
    print("por persona:", summary["per_signer"], "| por semilla:", summary["seeds"])
    print("matriz de confusión (filas = seña real):")
    print(cm)
    print(f"{time.time() - t0:.0f} s")

## 4. Entrenamiento final

Mismo procedimiento con los 1.125 clips. Se guarda en `artifacts/notebook_reproduction/bundle_reentrenado` y se compara archivo por archivo con `DELIVERED_BUNDLE` (por defecto `artifacts/iter3/bundle_iter4_v1/`, configurable con `LSC5_BUNDLE`). Si algún archivo difiere, la celda falla antes de la inferencia test.


In [ ]:
t0 = time.time()
RETRAINED = WORK / "bundle_reentrenado"
if RETRAINED.exists():
    shutil.rmtree(RETRAINED)
bundle = wave_iter.finalfit_v3(DATA_ROOT / "train", manifest, RETRAINED, cfg=CFG)
print(f"entrenado en {time.time() - t0:.0f} s; miembros:", [m["name"] for m in bundle["members"]], "| dim", bundle["members"][0]["dim"])
for f in sorted(p.name for p in DELIVERED_BUNDLE.glob("*") if p.suffix in (".pt", ".npz", ".npy")):
    identical = sha(RETRAINED / f) == sha(DELIVERED_BUNDLE / f)
    print(f"{f:22s}", "idéntico" if identical else "distinto")
    assert identical, f"El reentrenamiento difiere del artefacto fijado: {f}"

## Inferencia opcional con el modelo fijado

Este bloque se omite mientras `RUN_TEST_INFERENCE = False`. Con autorización y las rutas necesarias, carga los pesos originales con verificación de hashes, predice cada clip de test de forma independiente y escribe `artifacts/notebook_reproduction/submission.csv`. Compara el archivo con el CSV de referencia y las probabilidades con los pesos reentrenados; no ajusta ningún parámetro con test.


In [ ]:
if RUN_TEST_INFERENCE:
    assert (DATA_ROOT / "test" / "landmarks").is_dir(), "Faltan landmarks test autorizados"
    assert DELIVERED_CSV.is_file(), "Falta CSV de referencia autorizado"
    template = DATA_ROOT / "test" / "submission_template.csv"
    test_ids = [l.split(",")[0] for l in template.read_text(encoding="utf-8").splitlines()[1:] if l.strip()]
    test_paths = [DATA_ROOT / "test" / "landmarks" / f"{i}.npy" for i in test_ids]
    t0 = time.time()
    probs = wave_iter.predict_bundle_v3(DELIVERED_BUNDLE, test_paths, allow_test=True)
    print(f"{len(test_ids)} clips en {time.time() - t0:.1f} s")
    out_csv = WORK / "submission.csv"
    out_csv.unlink(missing_ok=True)
    check = write_submission(template, test_ids, probs, out_csv)
    print("CSV válido:", check)
    probs_retrained = wave_iter.predict_bundle_v3(RETRAINED, test_paths, allow_test=True)
    print("diferencia máxima de probabilidades, reentrenado frente a entregado:", float(np.abs(probs_retrained - probs).max()))
    print("coincide con submission.csv:", sha(out_csv) == sha(DELIVERED_CSV))
    print(out_csv.read_text(encoding="utf-8").splitlines()[:4])
    assert sha(out_csv) == sha(DELIVERED_CSV), "El CSV generado no coincide con la referencia"
    assert float(np.abs(probs_retrained - probs).max()) == 0.0, "Las probabilidades no coinciden"
else:
    print("Inferencia test omitida: RUN_TEST_INFERENCE = False")
